
**Corpus:** Official Hasbro **Monopoly** rulebook (8 pages) 
**Bi-Encoder:** `all-MiniLM-L6-v2` 
**Cross-Encoder:** `ms-marco-MiniLM-L-6-v2` 
**LLM:** Gemini `gemini-3.5-flash`

Two retrieval pipelines over the **same chunked corpus**:

| | Pipeline 1 — Baseline | Pipeline 2 — Re-Ranking |
|---|---|---|
| Stage 1 | Bi-Encoder → top-10 | Bi-Encoder → top-10 *(same 10)* |
| Stage 2 | take top-3 | **Cross-Encoder re-sorts** → new top-3 |
| Stage 3 | LLM → **Answer 1** | LLM *(same prompt)* → **Answer 2** |

Only the *selection of the 3 context chunks* differs between the pipelines — the corpus,
the retrieved-10 candidates and the prompt are identical, so the comparison is fair.

## 0. Setup

Install dependencies (uncomment on first run) and import libraries.

In [1]:
# !pip install pymupdf sentence-transformers faiss-cpu google-generativeai python-dotenv pandas numpy

import os, time, json, re
from pathlib import Path

import numpy as np
import pandas as pd
import pymupdf                      # PDF text extraction (PyMuPDF)
import faiss                        # vector store / nearest-neighbour search
from sentence_transformers import SentenceTransformer, CrossEncoder

pd.set_option("display.max_colwidth", 95)

# --- Configuration (all tunable knobs in one place) ---
PROJECT_ROOT   = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PDF_PATH       = PROJECT_ROOT / "data" / "monopoly_rules.pdf"
RESULTS_DIR    = PROJECT_ROOT / "results"; RESULTS_DIR.mkdir(exist_ok=True)

CHUNK_SIZE_WORDS    = 90           # words per chunk
CHUNK_OVERLAP_WORDS = 20           # overlap so a rule+exception isn't severed at a boundary
BI_ENCODER_MODEL    = "sentence-transformers/all-MiniLM-L6-v2"
CROSS_ENCODER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
TOP_K_RETRIEVE      = 10           # Stage-1 candidates
TOP_K_CONTEXT       = 3            # chunks actually shown to the LLM
GEMINI_MODEL        = "gemini-3.5-flash"
print("Project root:", PROJECT_ROOT)

/home/shravanikalmali/.local/lib/python3.12/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
/tmp/ipykernel_31786/138629112.py:7: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


/home/shravanikalmali/.local/lib/python3.12/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


Project root: /home/shravanikalmali/Desktop/sem7/LMA/assignment/LMA_A1


## Phase 1 - PDF Processing

Extract the text with PyMuPDF, normalise whitespace (but keep the wording so negations/exceptions survive), and split into **90-word chunks with 20-word overlap**.

Reason for choosign this chunk size - Small enough that a specific "except when…" clause lands in **its own chunk** (so it can be buried at rank 4-10 and later rescued), but large enough to stay coherent. The 20-word overlap stops a rule being cut in half at a boundary.

In [2]:
def extract_pages(pdf_path):
    """Return a list of per-page text strings."""
    doc = pymupdf.open(str(pdf_path))
    pages = [page.get_text() for page in doc]
    doc.close()
    return pages

def normalise(text):
    """Collapse whitespace/newlines; content words untouched."""
    text = text.replace("\xad", "")           # soft hyphens
    text = re.sub(r"[ \t]+", " ", text)        # collapse spaces/tabs
    text = re.sub(r"\n{2,}", "\n", text)       # collapse blank lines
    return text.strip()

def chunk_text(pages, chunk_size=CHUNK_SIZE_WORDS, overlap=CHUNK_OVERLAP_WORDS):
    """Word-based sliding window with overlap -> list of chunk dicts."""
    # Flatten into a (word, page_number) stream so each chunk can report its page.
    words_with_page = []
    for page_no, page_text in enumerate(pages, start=1):
        for w in normalise(page_text).split():
            words_with_page.append((w, page_no))

    step = max(1, chunk_size - overlap)
    chunks = []
    for start in range(0, len(words_with_page), step):
        window = words_with_page[start:start + chunk_size]
        if not window:
            break
        chunks.append({
            "chunk_id": len(chunks),
            "text": " ".join(w for w, _ in window),
            "pages": sorted({p for _, p in window}),
            "n_words": len(window),
        })
        if start + chunk_size >= len(words_with_page):
            break
    return chunks

corpus = chunk_text(extract_pages(PDF_PATH))
print(f"{len(corpus)} chunks, avg {np.mean([c['n_words'] for c in corpus]):.0f} words/chunk")
print("\n--- Example chunk #25 (contains the Free Parking rule) ---")
print(corpus[25]["text"])

42 chunks, avg 89 words/chunk

--- Example chunk #25 (contains the Free Parking rule) ---
Jail and immediately move forward the number of spaces shown by your throw. Even though you are in Jail, you may buy and sell property, buy and sell houses and hotels and collect rents. "FREE PARKING": A player landing on this place does not receive any money, property or reward of any kind. This is just a "free" resting place. HOUSES: When you own all the propert~es in a color-group you may buy houses from the Bank and erect them on those properties. If you buy one house, you may


## Phase 2 - Baseline Retrieval (Bi-Encoder + FAISS)

A **Bi-Encoder** maps the query and each chunk to a vector **independently**; relevance is the cosine similarity between them. Because chunk vectors are precomputed once and stored in a FAISS index, querying is a fast nearest-neighbour search.

We L2-normalise all vectors and use an inner-product index (`IndexFlatIP`), so **inner product = cosine similarity**, with exact (non-approximate) search over the 42 chunks.

In [3]:
# Load the Bi-Encoder and build the FAISS index (one-time cost).
bi_encoder = SentenceTransformer(BI_ENCODER_MODEL)

chunk_texts = [c["text"] for c in corpus]
chunk_emb = bi_encoder.encode(chunk_texts, convert_to_numpy=True,
                              show_progress_bar=False).astype("float32")
faiss.normalize_L2(chunk_emb)                       # unit vectors -> IP == cosine
index = faiss.IndexFlatIP(chunk_emb.shape[1])
index.add(chunk_emb)
print("Index built over", index.ntotal, "chunks,", index.d, "dims")

def bi_retrieve(query, top_k=TOP_K_RETRIEVE):
    """Return top_k chunks for `query`, annotated with bi_score (cosine) and bi_rank."""
    q = bi_encoder.encode([query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(q)
    scores, idxs = index.search(q, top_k)
    out = []
    for rank, (s, i) in enumerate(zip(scores[0], idxs[0]), start=1):
        c = dict(corpus[int(i)])
        c["bi_score"], c["bi_rank"] = float(s), rank
        out.append(c)
    return out

# Demo on a keyword-trap query
demo_q = "How large is the jackpot paid out to a player who lands on Free Parking?"
cands = bi_retrieve(demo_q)
pd.DataFrame([{"bi_rank": c["bi_rank"], "bi_score": round(c["bi_score"], 3),
               "preview": c["text"][:80]} for c in cands])

Index built over 42 chunks, 384 dims


,bi_rank,bi_score,preview
0,1,0.546,"the deck. The ""Get Out of Jail Free"" card is held until used and then returned t"
1,2,0.492,"paid only once each time kound the board. However, if a player passing GO on the"
2,3,0.491,"pertaining to the space on which you land. Retaining the dice, throw again and m"
3,4,0.466,"players and loans money when required on mortgages. The Bank collects all taxes,"
4,5,0.461,Jail and immediately move forward the number of spaces shown by your throw. Even
5,6,0.448,MONOPOW RULES OBJECT: The object of the game IS to become the wealthiest player
6,7,0.444,"0 0 ~ and $50~; 6 $40~; 5 each of $105, $ 5 ~ and $Is. All remaining money and o"
7,8,0.444,"to Jail"" space or by rolling doubles three times in a row) then your turn is ove"
8,9,0.416,NOCT property on which you will owe another player money. A few minor details: O
9,10,0.412,"of play land on that space, you are ""Just Visiting,"" you incur no penalty, and y"


## Phase 3 - Cross-Encoder Re-Ranking

A **Cross-Encoder** takes the (query, chunk) pair **together** (`[CLS] query [SEP] chunk [SEP]`) and runs full cross-attention, outputting a single relevance score per pair. It can tell that *"no rent can be collected"* actually answers a rent question, and that a keyword-overlapping chunk does not - distinctions the Bi-Encoder blurs.

Cost: one transformer pass **per pair**, so we only run it on the 10 candidates already shortlisted - never the whole corpus.

In [4]:
# Load the Cross-Encoder and define re-ranking of the SAME 10 candidates.
cross_encoder = CrossEncoder(CROSS_ENCODER_MODEL)

def ce_rerank(query, candidates):
    """Score each (query, chunk) jointly, re-sort descending, add ce_score/ce_rank."""
    pairs = [(query, c["text"]) for c in candidates]
    scores = cross_encoder.predict(pairs)
    out = [dict(c, ce_score=float(s)) for c, s in zip(candidates, scores)]
    out.sort(key=lambda c: c["ce_score"], reverse=True)
    for new_rank, c in enumerate(out, start=1):
        c["ce_rank"] = new_rank
    return out

reranked = ce_rerank(demo_q, cands)
pd.DataFrame([{"ce_rank": c["ce_rank"], "bi_rank": c["bi_rank"],
               "ce_score": round(c["ce_score"], 2), "preview": c["text"][:80]}
              for c in reranked])

,ce_rank,bi_rank,ce_score,preview
0,1,5,-2.89,Jail and immediately move forward the number of spaces shown by your throw. Even
1,2,2,-5.28,"paid only once each time kound the board. However, if a player passing GO on the"
2,3,3,-5.29,"pertaining to the space on which you land. Retaining the dice, throw again and m"
3,4,1,-5.69,"the deck. The ""Get Out of Jail Free"" card is held until used and then returned t"
4,5,8,-9.50,"to Jail"" space or by rolling doubles three times in a row) then your turn is ove"
5,6,6,-9.65,MONOPOW RULES OBJECT: The object of the game IS to become the wealthiest player
6,7,10,-10.00,"of play land on that space, you are ""Just Visiting,"" you incur no penalty, and y"
7,8,9,-10.44,NOCT property on which you will owe another player money. A few minor details: O
8,9,7,-10.90,"0 0 ~ and $50~; 6 $40~; 5 each of $105, $ 5 ~ and $Is. All remaining money and o"
9,10,4,-11.21,"players and loans money when required on mortgages. The Bank collects all taxes,"


Notice the correct **Free Parking** chunk (*"does not receive any money"*) jumps from a low Bi-Encoder rank up near the top after re-ranking - the keyword trap is undone.

## Phase 3b - LLM answer generation (shared prompt)

**Both pipelines use the identical prompt.** It forces the model to answer only from the supplied context and to abstain otherwise - so a retrieval miss shows up as an honest omission instead of a hallucination. The API key is read from `../.env`.

In [5]:
# Load GEMINI_API_KEY from ../.env
try:
    from dotenv import load_dotenv
    load_dotenv(PROJECT_ROOT / ".env")
except Exception:
    pass

PROMPT_TEMPLATE = """You are a precise assistant answering questions about the MONOPOLY board game rules.
Use ONLY the information in the context below. If the context does not contain
the answer, say "The provided rules do not specify this." Do not rely on outside
knowledge or popular house rules.

Context:
{context}

Question: {question}

Answer:"""

def build_prompt(question, chunks):
    ctx = "\n\n".join(f"[Chunk {i}] {c['text']}" for i, c in enumerate(chunks, 1))
    return PROMPT_TEMPLATE.format(context=ctx, question=question)

_gemini = {}
def generate_answer(question, chunks, max_retries=5):
    """Call Gemini on the given context. Retries on free-tier 429 (capped delay)."""
    import google.generativeai as genai
    from google.api_core import exceptions as gexc
    if "m" not in _gemini:
        genai.configure(api_key=os.environ["GEMINI_API_KEY"])
        _gemini["m"] = genai.GenerativeModel(GEMINI_MODEL)
    prompt = build_prompt(question, chunks)
    for attempt in range(max_retries):
        try:
            return _gemini["m"].generate_content(prompt).text.strip()
        except gexc.ResourceExhausted as e:
            delay = min(20 * (attempt + 1), 65)
            for d in getattr(e, "details", []) or []:
                secs = getattr(getattr(d, "retry_delay", None), "seconds", None)
                if isinstance(secs, int) and 0 < secs <= 120:
                    delay = secs + 2
            if attempt == max_retries - 1:
                raise
            print(f"  [rate-limited; waiting {delay}s...]"); time.sleep(delay)
print("Gemini generation configured (model:", GEMINI_MODEL + ")")

Gemini generation configured (model: gemini-3.5-flash)


## Phase 3c - Orchestration + timing helpers

`run_pipelines` runs **both** pipelines for one query and times each stage with `time.perf_counter()`. `locate_gold` finds the answer-bearing chunk (by a unique phrase) and its best rank before/after re-ranking; `rank_shift_table` shows how every candidate moved.

In [6]:
def run_pipelines(query, generate=True):
    t0 = time.perf_counter(); candidates = bi_retrieve(query); t_ret = time.perf_counter() - t0
    baseline_top3 = candidates[:TOP_K_CONTEXT]
    t0 = time.perf_counter(); reranked = ce_rerank(query, candidates); t_rr = time.perf_counter() - t0
    reranked_top3 = reranked[:TOP_K_CONTEXT]
    res = {"query": query, "candidates": candidates, "reranked": reranked,
           "baseline_top3": baseline_top3, "reranked_top3": reranked_top3,
           "timings": {"retrieve_s": t_ret, "rerank_s": t_rr}}
    if generate:
        res["answer_baseline"] = generate_answer(query, baseline_top3)
        res["answer_reranked"] = generate_answer(query, reranked_top3)
    return res

def locate_gold(res, gold_phrase, k=TOP_K_CONTEXT):
    """Best (smallest) rank among all chunks containing gold_phrase, before & after."""
    gp = gold_phrase.lower()
    bi = [c["bi_rank"] for c in res["candidates"] if gp in c["text"].lower()]
    ce = [c["ce_rank"] for c in res["reranked"]   if gp in c["text"].lower()]
    best_bi, best_ce = (min(bi) if bi else None), (min(ce) if ce else None)
    return {"bi_rank": best_bi, "ce_rank": best_ce,
            "in_baseline_top3": best_bi is not None and best_bi <= k,
            "in_reranked_top3": best_ce is not None and best_ce <= k}

def rank_shift_table(res):
    by_id = {c["chunk_id"]: c for c in res["reranked"]}
    rows = []
    for c in res["candidates"]:
        ce = by_id[c["chunk_id"]]
        rows.append({"chunk_id": c["chunk_id"], "bi_rank": c["bi_rank"], "ce_rank": ce["ce_rank"],
                     "shift": c["bi_rank"] - ce["ce_rank"],        # +ve = promoted
                     "bi_score": round(c["bi_score"], 3), "ce_score": round(ce["ce_score"], 2),
                     "preview": c["text"][:85]})
    return rows
print("helpers defined")

helpers defined


## Phase 4 - Failure-Mode Analysis (rank shifts)

Five stress-test queries, each targeting a known tricky passage (chosen empirically from ~25 candidates - on this small, well-written rulebook, failures have to be *engineered* with query vocabulary that differs from the rule's wording). `gold_phrase` is a substring unique to the genuinely-correct passage.

In [7]:
FAILURE_QUERIES = [
    {"id": "Q1", "category": "Keyword Trap / Distractor",
     "query": "How large is the jackpot paid out to a player who lands on Free Parking?",
     "gold_phrase": "does not receive any money"},
    {"id": "Q2", "category": "Negation / Constraint",
     "query": "Explain the rent payment rules for a property that has been mortgaged to the Bank.",
     "gold_phrase": "no rent can be collected"},
    {"id": "Q3", "category": "Exception / Corner Case",
     "query": "Can I sell a property to another player while it has houses on it?",
     "gold_phrase": "no property can be sold"},
    {"id": "Q4", "category": "Buried Relevant Chunk",
     "query": "In what order must houses be removed when I sell buildings back to the Bank?",
     "gold_phrase": "sold one by one"},
    {"id": "Q5", "category": "Honest Counter-Example (re-ranking hurts)",
     "query": "I rolled a normal number and stopped on the Jail square; do I owe any fine?",
     "gold_phrase": "incur no penalty"},
]

# Run retrieval + re-ranking for every query (no LLM yet -> fast).
results = {}
rows = []
for q in FAILURE_QUERIES:
    res = run_pipelines(q["query"], generate=False)
    results[q["id"]] = res
    g = locate_gold(res, q["gold_phrase"])
    if   not g["in_baseline_top3"] and g["in_reranked_top3"]: verdict = "RERANK FIXES"
    elif g["in_baseline_top3"] and not g["in_reranked_top3"]: verdict = "RERANK HURTS"
    elif g["in_baseline_top3"] and g["in_reranked_top3"]:     verdict = "both OK"
    else:                                                     verdict = "both FAIL"
    rows.append({"id": q["id"], "category": q["category"],
                 "gold_bi_rank": g["bi_rank"], "gold_ce_rank": g["ce_rank"],
                 "base_top3": g["in_baseline_top3"], "rr_top3": g["in_reranked_top3"],
                 "verdict": verdict})
pd.DataFrame(rows)

,id,category,gold_bi_rank,gold_ce_rank,base_top3,rr_top3,verdict
0,Q1,Keyword Trap / Distractor,5,1,False,True,RERANK FIXES
1,Q2,Negation / Constraint,4,1,False,True,RERANK FIXES
2,Q3,Exception / Corner Case,4,1,False,True,RERANK FIXES
3,Q4,Buried Relevant Chunk,5,3,False,True,RERANK FIXES
4,Q5,Honest Counter-Example (re-ranking hurts),2,4,True,False,RERANK HURTS


### Per-query rank-shift detail

`shift > 0` means the Cross-Encoder **promoted** that chunk. The ⭐ marks an answer-bearing (gold) chunk.

In [8]:
def show_shift(qid):
    q = next(x for x in FAILURE_QUERIES if x["id"] == qid)
    res = results[qid]; gp = q["gold_phrase"].lower()
    full = {c["chunk_id"]: c["text"].lower() for c in res["candidates"]}
    df = pd.DataFrame(rank_shift_table(res))
    df["gold"] = ["⭐" if gp in full[r] else "" for r in df["chunk_id"]]
    print(f"{qid}  [{q['category']}]\nQ: {q['query']}\ngold_phrase: {q['gold_phrase']!r}")
    return df.sort_values("bi_rank")

show_shift("Q1")

Q1  [Keyword Trap / Distractor]
Q: How large is the jackpot paid out to a player who lands on Free Parking?
gold_phrase: 'does not receive any money'


,chunk_id,bi_rank,ce_rank,shift,bi_score,ce_score,preview,gold
0,20,1,4,-3,0.546,-5.69,"the deck. The ""Get Out of Jail Free"" card is held until used and then returned to the",
1,15,2,2,0,0.492,-5.28,"paid only once each time kound the board. However, if a player passing GO on the thro",
2,14,3,3,0,0.491,-5.29,"pertaining to the space on which you land. Retaining the dice, throw again and move y",
3,11,4,10,-6,0.466,-11.21,"players and loans money when required on mortgages. The Bank collects all taxes, fine",
4,25,5,1,4,0.461,-2.89,Jail and immediately move forward the number of spaces shown by your throw. Even thou,⭐
5,8,6,6,0,0.448,-9.65,MONOPOW RULES OBJECT: The object of the game IS to become the wealthiest player throu,
6,9,7,9,-2,0.444,-10.90,"0 0 ~ and $50~; 6 $40~; 5 each of $105, $ 5 ~ and $Is. All remaining money and other",
7,7,8,5,3,0.444,-9.50,"to Jail"" space or by rolling doubles three times in a row) then your turn is over and",
8,6,9,8,1,0.416,-10.44,NOCT property on which you will owe another player money. A few minor details: Only t,
9,23,10,7,3,0.412,-10.00,"of play land on that space, you are ""Just Visiting,"" you incur no penalty, and you mo",


In [9]:
show_shift("Q2")

Q2  [Negation / Constraint]
Q: Explain the rent payment rules for a property that has been mortgaged to the Bank.
gold_phrase: 'no rent can be collected'


,chunk_id,bi_rank,ce_rank,shift,bi_score,ce_score,preview,gold
0,26,1,7,-6,0.606,-3.55,you may buy houses from the Bank and erect them on those properties. If you buy one h,
1,35,2,3,-1,0.588,1.29,"the mortgage from the Bank. However, the owner may sell this mortgaged property to an",
2,37,3,2,1,0.588,1.84,"making this settlement, if you own houses or hotels, you must retum these to the Bank",
3,33,4,1,3,0.584,4.49,reverse of the manner in which they were erected. MORTGAGES: Unimproved properties ca,⭐
4,38,5,5,0,0.564,-2.25,"is 10% of the value of the property. The new owner who does this may then, at hislher",
5,10,6,8,-2,0.542,-4.69,"the Bank. When more than f~ve persons play, the Banker may elect to act only as Banke",
6,34,7,4,3,0.510,0.96,can be collected on unmortgaged properties in the same group. In order to lift the mo,
7,39,8,9,-1,0.510,-5.17,"of taxes or penalties) even by selling off buildings and mortgaging property, you mus",
8,29,9,10,-1,0.508,-6.32,houses back to the Bank (see SELLING PROPERTY). HOTELS: When a player has four houses,
9,17,10,6,4,0.495,-2.83,"property. Any player, including the one who declined the option to buy it at the prin",⭐


In [10]:
show_shift("Q5")   # honest counter-example: re-ranking HURTS here

Q5  [Honest Counter-Example (re-ranking hurts)]
Q: I rolled a normal number and stopped on the Jail square; do I owe any fine?
gold_phrase: 'incur no penalty'


,chunk_id,bi_rank,ce_rank,shift,bi_score,ce_score,preview,gold
0,24,1,2,-1,0.548,-0.42,"take another turn; (2) using the ""Get Out of Jail Free"" card if you have it; (3) purc",
1,22,2,6,-4,0.537,-8.64,"you draw a card marked ""Go to JailN; or (3) you throw doubles three times in successi",⭐
2,21,3,7,-4,0.512,-9.14,"Bank, or you may pay 10% of your total worth to the Bank. Your total worth is all you",
3,6,4,1,3,0.473,-0.33,NOCT property on which you will owe another player money. A few minor details: Only t,
4,7,5,3,2,0.413,-5.64,"to Jail"" space or by rolling doubles three times in a row) then your turn is over and",
5,23,6,4,2,0.394,-7.21,"of play land on that space, you are ""Just Visiting,"" you incur no penalty, and you mo",⭐
6,20,7,8,-1,0.366,-9.44,"the deck. The ""Get Out of Jail Free"" card is held until used and then returned to the",
7,25,8,5,3,0.350,-8.61,Jail and immediately move forward the number of spaces shown by your throw. Even thou,
8,14,9,9,0,0.341,-9.46,"pertaining to the space on which you land. Retaining the dice, throw again and move y",
9,19,10,10,0,0.288,-10.77,to have houses or hotels on properties because rents are much higher than for unimpro,


## Phase 4b - LLM Answers (Answer 1 vs Answer 2)

Generate both answers with Gemini. To respect the free tier (~5 req/min, ~20/day **per model**), this cell **reuses cached answers** from `results/run_results.json` when present (set `REGENERATE = True` to force fresh, paced calls).

In [11]:
REGENERATE = False
CACHE = RESULTS_DIR / "run_results.json"
cache = json.loads(CACHE.read_text()) if (CACHE.exists() and not REGENERATE) else {}

first = True
for q in FAILURE_QUERIES:
    res = results[q["id"]]
    if q["id"] in cache:                                   # reuse saved answers
        a1, a2 = cache[q["id"]]["answer_baseline"], cache[q["id"]]["answer_reranked"]
    else:                                                   # live, paced calls
        if not first: time.sleep(14)
        a1 = generate_answer(q["query"], res["baseline_top3"]); time.sleep(14)
        a2 = generate_answer(q["query"], res["reranked_top3"]); first = False
    res["answer_baseline"], res["answer_reranked"] = a1, a2
    g = locate_gold(res, q["gold_phrase"])
    print("="*100)
    print(f"{q['id']} [{q['category']}]  gold {q['gold_phrase']!r}  (bi#{g['bi_rank']} -> ce#{g['ce_rank']})")
    print(f"Q: {q['query']}")
    print(f"\n-- Answer 1 (BASELINE top-3) --\n{a1}")
    print(f"\n-- Answer 2 (RE-RANKED top-3) --\n{a2}")

Q1 [Keyword Trap / Distractor]  gold 'does not receive any money'  (bi#5 -> ce#1)
Q: How large is the jackpot paid out to a player who lands on Free Parking?

-- Answer 1 (BASELINE top-3) --
The provided rules do not specify this.

-- Answer 2 (RE-RANKED top-3) --
Based on the provided rules, there
 is no jackpot. A player landing on Free Parking "does not receive any money, property or reward of any kind," as it is just a "free" resting place.
Q2 [Negation / Constraint]  gold 'no rent can be collected'  (bi#4 -> ce#1)
Q: Explain the rent payment rules for a property that has been mortgaged to the Bank.

-- Answer 1 (BASELINE top-3) --
The provided rules do not specify this.

-- Answer 2 (RE-RANKED top-3) --
Based on the provided rules, no rent can be collected on mortgaged properties or utilities. However, rent can still be collected on unmortgaged properties in the same color-group.
Q3 [Exception / Corner Case]  gold 'no property can be sold'  (bi#4 -> ce#1)
Q: Can I sell a property 

**Observations :**
- **Q1–Q3:** baseline **abstains** ("the provided rules do not specify this") because the gold chunk sat outside its top-3; re-ranking supplies it and Answer 2 is correct.
- **Q4:** both answer; re-ranking adds the precise *"one by one, evenly"* detail (completeness, not a correction).
- **Q5:** the reverse - baseline is correct, re-ranking drops the gold chunk and Answer 2 abstains. **Re-ranking is not a guarantee.**

## Phase 4c — Grounding probe: where does hallucination come from?

The rubric asks whether the baseline **hallucinates, omits, or misstates** when the critical chunk is missing. With our grounded prompt the baseline **omits** (abstains). But *why* didn't it hallucinate? This probe re-runs Q1's baseline context under two looser prompts.

**Finding:** `gemini-3.5-flash` is robust on a *famous* document — even when allowed to use outside knowledge, it answers Free Parking correctly (and even flags the "jackpot" as a house-rule myth), because the real rule is in its training data. So:

- On a **well-known** corpus like Monopoly, the LLM's *parametric memory* can mask a retrieval miss — which is exactly why we **force grounding** (`"use ONLY the context"`): it removes that crutch so the experiment measures *retrieval quality*, not memorisation. That is what turns the retrieval miss into a visible omission.
- On a **private / proprietary** corpus (where the LLM has no prior knowledge), the same retrieval miss under a permissive prompt would instead produce a **hallucination** — there is no correct memory to fall back on. That is the real-world danger re-ranking mitigates by getting the right chunk into context.

In [12]:
# Probe Q1's BASELINE context (which lacks the Free Parking rule) under looser prompts.
# Answers are cached in results/hallucination_demo.json (generated with gemini-3.5-flash).
q1 = next(x for x in FAILURE_QUERIES if x["id"] == "Q1")
ctx = "\n\n".join(f"[Chunk {i}] {c['text']}"
                   for i, c in enumerate(results["Q1"]["baseline_top3"], 1))

# (A) ungrounded: no "use ONLY the context / say you don't know" instruction
ungrounded_prompt = f"""You are an assistant answering questions about the MONOPOLY board game rules.
Use the context below to help answer.

Context:
{ctx}

Question: {q1['query']}

Answer:"""

# (B) permissive: explicitly allow the model's own general knowledge of Monopoly
permissive_prompt = f"""You are a helpful Monopoly assistant. Answer the player's question.
You may use the context below and your own general knowledge of Monopoly.

Context:
{ctx}

Question: {q1['query']}

Answer:"""

import json
hd = json.loads((RESULTS_DIR / "hallucination_demo.json").read_text()) \
     if (RESULTS_DIR / "hallucination_demo.json").exists() else {}
# To regenerate live instead of using cache, call generate_answer()-style on these prompts.

print("Q1 baseline context does NOT contain the Free Parking rule.\n")
print("(A) Ungrounded prompt ->", hd.get("baseline_answer_ungrounded"))
print("\n(B) Permissive (outside knowledge allowed) ->\n", hd.get("baseline_answer_permissive"))
print("\nConclusion: on a famous doc the model stays correct from memory; our grounded")
print("prompt removes that crutch so the retrieval miss shows up as omission (Answer 1).")

Q1 baseline context does NOT contain the Free Parking rule.

(A) Ungrounded prompt -> Based on the provided context, there is no mention of "Free Parking" or any jackpot paid for landing on it.

(B) Permissive (outside knowledge allowed) ->
 According to the official rules of Monopoly, **there is no jackpot** paid out when landing on Free Parking. 

Free Parking is simply a safe "resting" space where nothing happens—a player receives no money, properties, or rewards of any kind for landing there. 

*(Note: Collecting taxes, fees, or a jackpot on Free Parking is a very common "house rule," but it is not part of the official Monopoly rules!)*

Conclusion: on a famous doc the model stays correct from memory; our grounded
prompt removes that crutch so the retrieval miss shows up as omission (Answer 1).


## Phase 5 - Latency Profiling

Each stage is timed separately. We average over **6 measured passes × 5 queries** (2 warm-up passes discarded) for a stable mean. Generation is network-bound and dominates end-to-end.

In [13]:
ret_ms, rer_ms = [], []
for p in range(8):                       # 2 warm-up + 6 measured passes
    for q in FAILURE_QUERIES:
        res = run_pipelines(q["query"], generate=False)
        if p >= 2:
            ret_ms.append(res["timings"]["retrieve_s"]*1000)
            rer_ms.append(res["timings"]["rerank_s"]*1000)

gen_ms = [v["timings"].get("generate_baseline_s", 0)*1000 for v in cache.values()
          if v.get("timings", {}).get("generate_baseline_s")]
print(f"(averaged over {len(ret_ms)} samples)")
pd.DataFrame([
    {"stage": "Bi-Encoder retrieval (top-10)", "mean_ms": round(np.mean(ret_ms), 2),
     "note": "encode query + FAISS search; precomputed index"},
    {"stage": "Cross-Encoder re-rank (10 pairs)", "mean_ms": round(np.mean(rer_ms), 2),
     "note": f"~{np.mean(rer_ms)/np.mean(ret_ms):.1f}x retrieval; scales with #candidates"},
    {"stage": "LLM generation (per answer)", "mean_ms": round(np.mean(gen_ms), 0) if gen_ms else None,
     "note": "network-bound (~seconds); dominates end-to-end"},
])

(averaged over 30 samples)


,stage,mean_ms,note
0,Bi-Encoder retrieval (top-10),2.85,encode query + FAISS search; precomputed index
1,Cross-Encoder re-rank (10 pairs),14.61,~5.1x retrieval; scales with #candidates
2,LLM generation (per answer),NaN,network-bound (~seconds); dominates end-to-end


In [14]:
# Baseline pipeline vs two-stage pipeline (retrieval side)
pd.DataFrame([
    {"pipeline": "Baseline (retrieve only)", "retrieval_ms": round(np.mean(ret_ms), 2),
     "rerank_ms": 0.0, "total_ms": round(np.mean(ret_ms), 2)},
    {"pipeline": "Re-ranking (retrieve + rerank)", "retrieval_ms": round(np.mean(ret_ms), 2),
     "rerank_ms": round(np.mean(rer_ms), 2), "total_ms": round(np.mean(ret_ms)+np.mean(rer_ms), 2)},
])

,pipeline,retrieval_ms,rerank_ms,total_ms
0,Baseline (retrieve only),2.85,0.00,2.85
1,Re-ranking (retrieve + rerank),2.85,14.61,17.46


## Scaling question - why not Cross-Encode the whole corpus?

The Bi-Encoder embeds every chunk **once** and reuses those vectors forever, so a query costs ~1 encode + a fast vector search → scales to millions of chunks. The Cross-Encoder score depends on the **specific (query, chunk) pair**, so it **cannot be precomputed**: every query needs **one transformer pass per chunk**. Scoring 10 candidates already costs ~4–5× the whole retrieval step; scoring a 10k-chunk corpus would be ~1000× → seconds per query. Hence **retrieve-then-rerank**: the cheap model narrows millions → 10, the expensive model refines only those 10. (Limitation: the re-ranker can't recover a gold chunk that never made the top-10 - its benefit depends on first-stage recall.)

In [15]:
# (Optional) persist this run's answers + timings for the report
out = {qid: {"query": r["query"],
             "answer_baseline": r.get("answer_baseline"),
             "answer_reranked": r.get("answer_reranked"),
             "timings": r["timings"]} for qid, r in results.items()}
(RESULTS_DIR / "run_results.json").write_text(json.dumps(out, indent=2, default=str))
print("saved results/run_results.json")

saved results/run_results.json
